# ACHI physiology analysis: stress → HRV / EDA → prior weighting

This notebook runs the pipeline in `achi_pipeline/`, which implements **`script_outline_REVISED.md`** (your current script outline, with notes where the code had to make a choice).

| Stage | What | Outline step |
|---|---|---|
| 0 | Check inputs, fill in the event names in `config.py` | 1 |
| 1 | Load, clean ECG + EDA, cut segments, write review files | 2, 3 |
| 2 | **Visual check** of every flag; segment keep/exclude decisions | 4 |
| 3 | RMSSD, SCL, SCR frequency, change scores, recovery time, FP rates, ν | 5 |
| 4 | Participant table (one row per participant), run table, statistics, descriptives | 6, 7 |

Set `USE_SYNTHETIC` in the first code cell to choose synthetic or real data. Outputs go to `synthetic_derived/` or `derived/`. Problems are logged in `derived/logs/missing_and_errors.txt`.
Stage 1 can be re-run safely: review decisions you've already entered are kept.

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from IPython.display import Image, display
from achi_pipeline import config, pipeline, review, hgf
pd.set_option("display.max_columns", 60)

# True  = practise on the synthetic participants in synthetic_data/ (make them with make_synthetic_data.py)
# False = real data in ../Data/ (read only) ; outputs go to derived/
USE_SYNTHETIC = True
config.use_synthetic(USE_SYNTHETIC)

print("data:", config.DATA_DIR, "| exists:", config.DATA_DIR.exists())
print("derived:", config.DERIVED_DIR)

## Stage 0: inputs

Each participant needs `ACHI_2026_XXX_session-1_sessiondata.json` (the session log, with the word-detection trials inside). The `.acq` recording is optional for now: without it the ECG/EDA steps are skipped and logged, while FP rates and ν are still computed. `participants.csv` (in the data folder) needs the columns `participant_id, group, age, antipsychotic` (group = patient / control).

`inspect_participant` prints every session-log event per run with its duration, the run types, missing Trial-TT blocks, a trial summary, and (with a recording) how the trigger blocks were matched.

**Before real data:** fill in the stress vs non-stress rule (`RUN_TYPE_RULE` in `achi_pipeline/config.py`, or `run_types.csv` in the data folder). Until then real MIST runs are `unknown`.

In [ ]:
found = pipeline.discover_participants()
pd.DataFrame([{"participant_id": k, **{t: len(v[t]) for t in ("acq", "json")}} for k, v in found.items()])

Get detailed report about specific participant

In [ ]:
if found:
    pipeline.inspect_participant("ACHI_2026_906")

## Stage 1: preprocessing (all participants)

In [ ]:
stage1 = pipeline.run_all_preprocessing()      # or pipeline.run_all_preprocessing(["ACHI_2026_003"])
stage1

In [ ]:
pid = next(iter(found)) if found else None      # participant to look at / review
if pid:
    display(Image(str(pipeline.participant_dir(pid) / "qc_overview.png")))
    display(pd.read_csv(pipeline.participant_dir(pid) / "segments.csv"))

## Stage 2: visual check (outline 4)

For every flag, compare the raw and cleaned signals and choose a decision:

* **ECG**: `accept` (detection/correction is right), `add_peak` / `move_peak` (type the peak time in *new time s*), `remove_peak`, or `exclude_span` (unusable signal; type the end time, or `start-end`, in *new time s*).
* **EDA**: `accept` (it is an artifact; masked from SCL) or `not_artifact`.

Then record a keep/exclude decision and a reason for any segment you want excluded.
* **ECG** segments where artefacts or corrected beats cover more than **20% of the duration** are excluded automatically (`pct_ecg_bad`).
* **EDA**: flagged portions are removed from SCL and SCR counts (`pct_eda_removed`); a whole segment is only excluded if you decide so, with a reason.
You can also edit the CSVs in `derived/participants/<pid>/review/` directly.

In [ ]:
app = review.ReviewApp(pid)
app.show("ecg")

In [ ]:
app.show("eda")

In [ ]:
print(review.review_status(pid))      # (reviewed, total) per flag file
app.segment_table()

In [ ]:
# Example: exclude one segment's ECG for a stated reason
# app.set_segment_decision("run-2", "task", "ecg", "exclude", "electrode came off at 12:05")

### Practice mode (synthetic data only)

With `USE_SYNTHETIC = True` the true heartbeats and injected artefacts are known, so you can check your review:

* `simulated_review.score(pid)` compares your decisions with what a careful reviewer should decide, flag by flag, and says why.
* `simulated_review.apply(pid)` lets a **simulated reviewer** fill in a participant's sheets (with ~5% overlooked corrections, as a real person would make). Use it for the participants you don't want to review by hand.

Practise tips. Each synthetic participant contains:
- missed beats (no QRS drawn: keep the interpolated beat);
- sharp spikes (remove the peak);
- peaks the algorithm shifted off the R wave (move them back);
- a movement burst (exclude the span: type `start-end` in *new time s*).

Some also contain an ECG dropout, EDA jumps or lost contact, and genuine large SCRs that should be marked `not_artifact`.

In [ ]:
if USE_SYNTHETIC:
    from achi_pipeline import simulated_review
    result = simulated_review.score(pid)                      # how did you do?
    display(result[~result["correct"]])                       # the flags you got 'wrong', with the reason

In [ ]:
if USE_SYNTHETIC:
    # let the simulated reviewer do every participant you haven't fully reviewed yourself
    for p in pipeline.discover_participants():
        if not review.is_reviewed(p):
            log = simulated_review.apply(p, error_rate=0.05)
            print(p, "reviewed by simulation:", log["decision"].value_counts().to_dict())

## Stage 3: measures

Only participants with every flag reviewed are processed. For a provisional look before review, set `config.ALLOW_UNREVIEWED = True`; those rows are marked `reviewed = False`.

ν comes from `hgf.py` unless `nu_estimates.csv` (from TAPAS) is present. Check parameter recovery first (cell below).
By default only silent-trial responses enter the ν fit (`config.HGF_LIKELIHOOD_TRIALS = "silent"`, as in the outline); ν is recovered much better with `"all"`.

In [ ]:
# config.ALLOW_UNREVIEWED = True
stage3 = pipeline.run_all_compute()
{k: (None if v is None else len(v["runs"])) for k, v in stage3.items()}     # runs per participant

In [ ]:
# Optional: can the Python HGF recover nu with a task like ours? (takes ~10 s; uses config.HGF_LIKELIHOOD_TRIALS)
rec = hgf.parameter_recovery(nus=(0.2, 0.4, 0.6, 0.8), n_sims=4, n_trials=120)
rec.groupby("true_nu")["nu"].agg(["mean", "std"])

## Stage 4: statistics and descriptive summary (outline 6–7)

In [ ]:
res = pipeline.analyse()
res["table"]          # participant table: one row per participant

In [ ]:
res["runs"]           # run table: one row per run (order comparison, SCR by run type)

In [ ]:
cols = ["test", "family", "model", "n_participants", "scale", "estimate", "ci_low", "ci_high", "p", "significant", "note"]
res["statistics"][cols]

In [ ]:
res["descriptives"]

In [ ]:
res["descriptives_scr"]

In [ ]:
res["ecg_log"]        # % corrected beats per participant and segment (also in logs/ecg_corrected_beats.csv)

In [ ]:
res["supplementary"]